# 04. Ensembles: diversity, the candidates and the finalist check

**Project:** Kaggle House Prices. **Stage:** after modelling (notebook 03).

Thin notebook: it displays the base-model diversity diagnostic, the registered ensembles E068 to E075 and the finalist check on fresh folds, all from files produced by `experiments/ensemble/run_ensembles.py` and `experiments/finalists/run_finalists.py` and from the registry. Nothing is refit and the holdout is not used.

The finalist check re-scores each candidate with RepeatedKFold(5 x 3) on seed 7, folds that share nothing with the seed-42 selection folds (ruling D17), so its mean is not selection-biased.

In [ ]:
# Resolve the project root the same way notebook 01 does: walk up until data_description.txt is found.
import os
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "data_description.txt").exists():
    if root == root.parent:
        raise FileNotFoundError("Project root (data_description.txt) not found")
    root = root.parent
os.chdir(root)
print("Working directory:", Path.cwd())

In [ ]:
import io
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from houseprices import registry

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
warnings.filterwarnings("ignore", category=FutureWarning)

def paired(a_scores, b_scores):
    d = np.asarray(a_scores, float) - np.asarray(b_scores, float)
    return d.mean(), d.std(ddof=1) / np.sqrt(len(d))

## 1. Diversity of the base models

Residual correlation of single-pass out-of-fold predictions (diagnostic only). Ensembles help when errors are different: a linear model and a tree correlate at about 0.82 to 0.86, two linear models at 0.97 or more.

In [ ]:
text = open("experiments/ensemble/diversity_20261116_234858.csv", encoding="utf-8").read()
blocks = {}
for chunk in text.split("# ")[1:]:
    header, _, body = chunk.partition("\n")
    blocks[header.strip()] = pd.read_csv(io.StringIO(body), index_col=0)
display(blocks["diversity_report"].round(4))
rc = blocks["residual_correlation"]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(rc.to_numpy(), vmin=0.8, vmax=1.0, cmap="viridis")
ax.set_xticks(range(len(rc))); ax.set_xticklabels(rc.columns, rotation=45, ha="right")
ax.set_yticks(range(len(rc))); ax.set_yticklabels(rc.index)
for i in range(len(rc)):
    for j in range(len(rc)):
        ax.text(j, i, f"{rc.iat[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
plt.colorbar(im, label="residual correlation")
ax.set_title("Residual correlation of the six bases")
plt.tight_layout(); plt.show()

## 2. The registered ensembles (E068 to E075)

Selection-fold scores. The last columns compare each ensemble with the single CatBoost E066 on the same five folds: mean of the per-fold differences and its standard error (difference / SE). Negative means the ensemble is better; the project asks for more than 0.003 or about one SE before calling a difference a win.

In [ ]:
reg = registry.load_registry().set_index("id")
base = reg.loc["E066", "fold_scores"]
rows = []
for eid in ["E066"] + [f"E{n:03d}" for n in range(68, 76)]:
    r = reg.loc[eid]
    d, se = paired(r["fold_scores"], base)
    rows.append({"id": eid, "title": r["title"].replace("Ensemble: ", ""), "cv_mean": r["cv_mean"], "cv_std": r["cv_std"],
                 "diff vs E066": d, "paired SE": se})
display(pd.DataFrame(rows).round(5))

## 3. The finalists on fresh folds (RepeatedKFold 5 x 3, seed 7)

`selection_cv_mean` is the registered seed-42 score, `fresh_cv_mean` the seed-7 re-score; `optimism` is their difference, the selection bias. The paired columns are recomputed here from the registry fold scores (15 folds) against the finalist record of E066; the 15 folds come from three repeats of the same rows, so the standard error is slightly too small and should be read loosely.

In [ ]:
fin = pd.read_csv("experiments/finalists/finalists_20261117_003230.csv")
ref_id = fin.loc[fin["source_id"] == "E066", "finalist_id"].iloc[0]
ref_scores = reg.loc[ref_id, "fold_scores"]
diffs = []
for fid in fin["finalist_id"]:
    d, se = paired(reg.loc[fid, "fold_scores"], ref_scores)
    diffs.append((d, se))
fin["diff vs E066 (fresh)"] = [d for d, _ in diffs]
fin["paired SE"] = [se for _, se in diffs]
cols = ["source_id", "finalist_id", "title", "selection_cv_mean", "fresh_cv_mean", "optimism", "diff vs E066 (fresh)", "paired SE"]
display(fin[cols].assign(title=fin["title"].str.replace("Ensemble: ", "")).round(5))
print("Reference finalist record for E066:", ref_id)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = fin.sort_values("fresh_cv_mean")
ax.barh(order["source_id"] + " " + order["title"].str.replace("Ensemble: ", "").str[:34], order["fresh_cv_mean"], color="#4c78a8")
ax.axvline(reg.loc[ref_id, "cv_mean"], color="#e45756", ls="--", label="E066 fresh mean")
ax.set_xlim(order["fresh_cv_mean"].min() - 0.004, order["fresh_cv_mean"].max() + 0.002)
ax.invert_yaxis(); ax.set_xlabel("fresh-fold CV RMSE (seed 7, 5 x 3)"); ax.legend()
plt.tight_layout(); plt.show()

Reading: every finalist loses 0.0027 to 0.0044 between selection and fresh folds, so the registered numbers are optimistic by that amount. The ensembles that mix a linear model and a tree differ from the single CatBoost by 0.001 or less, which is inside the noise band (see the paired SE). The finalist ruling is recorded in `docs/EXPERIMENT_HISTORY.md` (D22).